# Integer & Nonlinear Portfolio Optimization: Live Market Data
**OPIM 5641 — Extended Assignment**  
*Generalized from the Womack MINLP example to real-world equity data*

---

This notebook extends the Womack portfolio problem into a fully generalized, live-data MINLP optimizer.

**What's new vs. the Womack template:**
- Live ticker data via `yfinance` (2 years of monthly returns, pulled at runtime)
- Dynamic sector classification via `yfinance` metadata — no hardcoding
- 10 representative stocks across 5 sectors (2 per sector)
- Expanded constraint set: sector caps, defensive floor, beta ceiling, cardinality, contingency logic
- Full constraint validation report
- Efficient frontier with integer discontinuity annotation

**Solver:** Bonmin (COIN-OR MINLP via IDAES)


In [ ]:
%%capture
import sys, os

# Install IDAES + Bonmin (Colab only)
if 'google.colab' in sys.modules:
    !pip install idaes-pse --pre -q
    !idaes get-extensions --to ./bin
    os.environ['PATH'] += ':bin'
    !pip install pyomo yfinance -q
else:
    # Local: make sure bonmin is on your PATH
    pass


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

from pyomo.environ import *
from pyomo.opt import SolverStatus, TerminationCondition

print("All imports OK.")


## Step 1: Define the Ticker Universe

We pick **10 tickers** spanning 5 sectors — 2 per sector.  
Sectors were chosen to ensure meaningful diversification constraints:  
`Technology`, `Healthcare`, `Financials`, `Energy`, `Consumer Staples`

Sector labels are pulled **live from yfinance metadata**, not hardcoded.  
This means if a company reclassifies, the model adapts automatically.


In [ ]:
# ── 10 Representative Tickers (2 per sector) ──────────────────────────────
TICKERS = [
    "AAPL",  # Technology
    "MSFT",  # Technology
    "JNJ",   # Healthcare
    "UNH",   # Healthcare
    "JPM",   # Financials
    "BAC",   # Financials
    "XOM",   # Energy
    "CVX",   # Energy
    "PG",    # Consumer Staples
    "KO",    # Consumer Staples
]

# ── Pull 2 years of monthly price history ─────────────────────────────────
print("Downloading price data from Yahoo Finance...")
raw = yf.download(TICKERS, period="2y", interval="1mo", auto_adjust=True, progress=False)["Close"]
raw = raw.dropna()

# Convert to monthly returns
returns_df = raw.pct_change().dropna()
returns_df = returns_df[TICKERS]  # enforce column order

print(f"  Periods (months):  {len(returns_df)}")
print(f"  Tickers loaded:    {len(returns_df.columns)}")
print()
returns_df.head()


## Step 2: Dynamic Sector Classification

We query `yfinance` for each ticker's sector at runtime.  
This drives all sector-level constraints — no manual mapping required.


In [ ]:
# ── Pull sector for each ticker via yfinance metadata ────────────────────
print("Fetching sector classifications...")
sector_map = {}  # ticker -> sector string
for t in TICKERS:
    info = yf.Ticker(t).info
    sector_map[t] = info.get("sector", "Unknown")
    print(f"  {t:6s}  ->  {sector_map[t]}")

# Invert: sector -> list of tickers
from collections import defaultdict
sector_groups = defaultdict(list)
for ticker, sector in sector_map.items():
    sector_groups[sector].append(ticker)

print()
print("Sector groups:")
for s, tks in sector_groups.items():
    print(f"  {s}: {tks}")


## Step 3: Summary Statistics

Mean monthly return and standard deviation for each stock.  
High return + high volatility stocks are the ones where the integer
constraints will matter most — they'll get selected or blocked based on logic rules.


In [ ]:
df_return = returns_df.mean()
df_std    = returns_df.std()
df_cov    = returns_df.cov()

summary = pd.DataFrame({
    "Mean Monthly Return": df_return,
    "Std Dev (Risk)":      df_std,
    "Sector":              pd.Series(sector_map)
}).sort_values("Mean Monthly Return", ascending=False)

print(summary.to_string())

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
summary["Mean Monthly Return"].plot(kind='bar', ax=axes[0], color='steelblue', alpha=0.85)
axes[0].set_title("Mean Monthly Return by Stock")
axes[0].set_ylabel("Return")
axes[0].tick_params(axis='x', rotation=45)

summary["Std Dev (Risk)"].plot(kind='bar', ax=axes[1], color='tomato', alpha=0.85)
axes[1].set_title("Std Dev (Volatility) by Stock")
axes[1].set_ylabel("Std Dev")
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
# Covariance matrix heatmap
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(df_cov.values, cmap='RdYlGn_r', aspect='auto')
ax.set_xticks(range(len(TICKERS))); ax.set_xticklabels(TICKERS, rotation=45, ha='right')
ax.set_yticks(range(len(TICKERS))); ax.set_yticklabels(TICKERS)
plt.colorbar(im, ax=ax, label='Covariance')
ax.set_title("Return Covariance Matrix\n(negative covariance = natural hedge)")
plt.tight_layout()
plt.show()


## Step 4: Constraint Design

This is where we extend beyond the Womack template. Every constraint below
has an economic rationale — not just a textbook exercise.

| # | Constraint | Type | Rationale |
|---|-----------|------|-----------|
| 1 | Weights sum to 1 | Equality | Fully invested portfolio |
| 2 | Each weight: 5% floor, 40% cap | Linking | Avoid trivial/over-concentrated positions |
| 3 | Cardinality ≤ 5 stocks | Integer | Transaction cost / management overhead |
| 4 | Cardinality ≥ 2 stocks | Integer | Minimum diversification |
| 5 | No single sector > 40% | Sector cap | Sector concentration risk |
| 6 | At least 1 defensive sector held | Integer | Portfolio must include Healthcare OR Consumer Staples |
| 7 | Contingency: if Tech, must hold Financials | Integer | Tech runs on credit; hedge correlated macro risk |
| 8 | Mutual exclusion: not both pure Energy names | Integer | XOM + CVX are highly correlated; pick one |
| 9 | Risk (portfolio variance) ≤ threshold | Quadratic | The efficient frontier sweep parameter |

Constraints 1-8 are constant across the frontier sweep.  
Constraint 9 is the parameter we vary to trace the efficient frontier.


## Step 5: Build the MINLP Model

The model structure mirrors the Womack fixed version but is fully generalized:
- Variables and constraints built from `TICKERS` list — add/remove a ticker and everything updates
- Sector constraints constructed from the live `sector_groups` dict
- Risk expression computed from the live covariance matrix


In [ ]:
def solve_portfolio(r_limit, verbose=False):
    """
    Solve the MINLP portfolio problem for a given risk (variance) limit.
    Returns: (allocation_dict, return_value) or (None, None) if infeasible.
    """
    m = ConcreteModel()

    # ── Decision Variables ─────────────────────────────────────────────────
    # x[t]: continuous weight (proportion) in stock t
    # y[t]: binary — 1 if we invest in stock t, 0 otherwise
    m.x = Var(TICKERS, domain=NonNegativeReals, bounds=(0, 1))
    m.y = Var(TICKERS, domain=Binary)

    # ── Objective: Maximize expected return ────────────────────────────────
    m.obj = Objective(
        expr=sum(m.x[t] * df_return[t] for t in TICKERS),
        sense=maximize
    )

    # ── Constraint 1: Fully invested (weights sum to 1) ────────────────────
    m.sum_weights = Constraint(expr=sum(m.x[t] for t in TICKERS) == 1)

    # ── Constraints 2a/2b: Position floor (5%) and cap (40%) via linking ───
    # x[t] <= 0.40 * y[t]  =>  if y=0, x must be 0; if y=1, x <= 0.40
    # x[t] >= 0.05 * y[t]  =>  if y=1, x must be at least 5%
    m.link_hi = Constraint(TICKERS, rule=lambda m, t: m.x[t] <= 0.40 * m.y[t])
    m.link_lo = Constraint(TICKERS, rule=lambda m, t: m.x[t] >= 0.05 * m.y[t])

    # ── Constraint 3: Maximum 5 stocks held ───────────────────────────────
    m.card_max = Constraint(expr=sum(m.y[t] for t in TICKERS) <= 5)

    # ── Constraint 4: Minimum 2 stocks held ───────────────────────────────
    m.card_min = Constraint(expr=sum(m.y[t] for t in TICKERS) >= 2)

    # ── Constraint 5: No sector > 40% of portfolio (dynamic) ──────────────
    # Built from live sector_groups — generalizes to any ticker universe
    m.sector_cap = ConstraintList()
    for sector, tks in sector_groups.items():
        if len(tks) > 1:  # only meaningful if we have 2+ in a sector
            m.sector_cap.add(sum(m.x[t] for t in tks) <= 0.40)

    # ── Constraint 6: Must hold at least 1 defensive sector stock ─────────
    # Healthcare OR Consumer Staples must appear in the portfolio
    defensive_tickers = (
        sector_groups.get("Healthcare", []) +
        sector_groups.get("Consumer Staples", [])
    )
    if defensive_tickers:
        m.defensive = Constraint(
            expr=sum(m.y[t] for t in defensive_tickers) >= 1
        )

    # ── Constraint 7: Contingency — if Tech, must hold Financials ─────────
    tech_tickers = sector_groups.get("Technology", [])
    fin_tickers  = sector_groups.get("Financials", [])
    if tech_tickers and fin_tickers:
        # sum(y_tech) <= |fin| * sum(y_fin)
        # i.e. you can't have any tech unless at least one financial is held
        m.tech_fin_contingency = Constraint(
            expr=sum(m.y[t] for t in tech_tickers) <=
                 len(fin_tickers) * sum(m.y[t] for t in fin_tickers)
        )

    # ── Constraint 8: Mutual exclusion — XOM and CVX can't both be held ───
    if "XOM" in TICKERS and "CVX" in TICKERS:
        m.energy_mutex = Constraint(expr=m.y["XOM"] + m.y["CVX"] <= 1)

    # ── Constraint 9: Risk ceiling (quadratic) — swept over frontier ───────
    risk_expr = sum(
        m.x[t1] * df_cov.at[t1, t2] * m.x[t2]
        for t1 in TICKERS for t2 in TICKERS
    )
    m.risk_limit = Constraint(expr=risk_expr <= r_limit)

    # ── Solve ──────────────────────────────────────────────────────────────
    solver = SolverFactory('bonmin', executable='/content/bin/bonmin')
    result = solver.solve(m, tee=verbose)

    tc = result.solver.termination_condition
    if tc == TerminationCondition.optimal:
        alloc = {t: value(m.x[t]) for t in TICKERS}
        ret   = value(m.obj)
        return alloc, ret
    return None, None

print("Model function defined. Ready to sweep the efficient frontier.")


## Step 6: Sweep the Efficient Frontier

We solve the MINLP at each risk level from very low to very high.  
At low risk, the optimizer is forced into conservative, diversified positions.  
At high risk, it concentrates into the highest-return names (subject to integer rules).

Because we have binary variables, the frontier will show **discrete jumps** — points where
adding a new stock or dropping a defensive position unlocks a higher return tier.
These jumps don't exist in the continuous MPT solution. That's the whole point of the integer formulation.


In [ ]:
# Risk range — start tight (forced diversification) and open up
risk_range = np.arange(0.0005, 0.0060, 0.0002)

print(f"Sweeping {len(risk_range)} risk levels via Bonmin...")
print("-" * 50)

frontier_results = []

for r in risk_range:
    alloc, ret = solve_portfolio(r)
    if alloc is not None:
        row = alloc.copy()
        row['Risk']   = r
        row['Return'] = ret
        n_held = sum(1 for t in TICKERS if alloc[t] > 1e-4)
        row['N_Stocks'] = n_held
        frontier_results.append(row)
        print(f"  Risk ≤ {r:.4f}  |  Return = {ret:.4f}  |  Stocks held: {n_held}")
    else:
        print(f"  Risk ≤ {r:.4f}  |  INFEASIBLE")

print()
print(f"Feasible solutions: {len(frontier_results)} / {len(risk_range)}")
res_df = pd.DataFrame(frontier_results).set_index('Risk')


## Step 7: Visualize Results

In [ ]:
if len(res_df) == 0:
    print("No feasible solutions — check solver path or constraints.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle("Integer Portfolio Optimization — Live Market Data", fontsize=14, fontweight='bold')

    # ── Plot 1: Efficient Frontier ─────────────────────────────────────────
    ax = axes[0, 0]
    sc = ax.scatter(res_df.index, res_df['Return'],
                    c=res_df['N_Stocks'], cmap='plasma', s=60, zorder=5)
    ax.plot(res_df.index, res_df['Return'], 'k-', alpha=0.3, linewidth=1)
    plt.colorbar(sc, ax=ax, label='Stocks Held')
    ax.set_title("Efficient Frontier\n(color = # stocks in portfolio)")
    ax.set_xlabel("Portfolio Risk (Variance)")
    ax.set_ylabel("Expected Monthly Return")
    ax.grid(True, alpha=0.2)

    # ── Plot 2: Stacked area allocation ───────────────────────────────────
    ax = axes[0, 1]
    res_df[TICKERS].plot(kind='area', stacked=True, ax=ax, alpha=0.75, cmap='tab10')
    ax.set_title("Portfolio Allocation by Risk Level")
    ax.set_xlabel("Portfolio Risk (Variance)")
    ax.set_ylabel("Weight")
    ax.legend(loc='upper right', fontsize=7, ncol=2)

    # ── Plot 3: Sector composition across frontier ─────────────────────────
    ax = axes[1, 0]
    sector_weights = pd.DataFrame(index=res_df.index)
    for sector, tks in sector_groups.items():
        valid_tks = [t for t in tks if t in res_df.columns]
        sector_weights[sector] = res_df[valid_tks].sum(axis=1)
    sector_weights.plot(kind='area', stacked=True, ax=ax, alpha=0.75, cmap='Set2')
    ax.set_title("Sector Allocation by Risk Level")
    ax.set_xlabel("Portfolio Risk (Variance)")
    ax.set_ylabel("Sector Weight")
    ax.legend(loc='upper right', fontsize=8)

    # ── Plot 4: Cardinality across frontier ────────────────────────────────
    ax = axes[1, 1]
    ax.step(res_df.index, res_df['N_Stocks'], where='post', color='navy', linewidth=2)
    ax.fill_between(res_df.index, res_df['N_Stocks'], step='post', alpha=0.15, color='navy')
    ax.axhline(5, color='red',   linestyle='--', alpha=0.5, label='Max (5)')
    ax.axhline(2, color='green', linestyle='--', alpha=0.5, label='Min (2)')
    ax.set_title("Number of Stocks Held by Risk Level\n(step function = integer constraint effect)")
    ax.set_xlabel("Portfolio Risk (Variance)")
    ax.set_ylabel("# Stocks Held")
    ax.set_ylim(0, 7)
    ax.legend()
    ax.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()


## Step 8: Constraint Validation Report

Same approach as the Womack template but extended to cover all 8 constraint categories.  
Every row in `res_df` represents one solved portfolio — we verify each one passes all rules.


In [ ]:
epsilon = 1e-4  # numerical tolerance for solver residues

print("=" * 55)
print("   PORTFOLIO CONSTRAINT VALIDATION REPORT")
print("=" * 55)
print(f"   Risk levels analyzed:      {len(res_df)}")
print()

# 1. Weights sum to 1
weight_sums = res_df[TICKERS].sum(axis=1)
bad_sums = (weight_sums < 0.999) | (weight_sums > 1.001)
print(f"[C1] Sum-to-1 violations:        {bad_sums.sum()}")

# 2a. Floor (5%) violations
is_invested = res_df[TICKERS] > epsilon
floor_viol = (is_invested) & (res_df[TICKERS] < 0.0499)
print(f"[C2a] Floor (<5%) violations:    {floor_viol.sum().sum()}")

# 2b. Cap (40%) violations
cap_viol = res_df[TICKERS] > 0.401
print(f"[C2b] Cap (>40%) violations:     {cap_viol.sum().sum()}")

# 3. Cardinality max
card_max_viol = (is_invested.sum(axis=1) > 5).sum()
print(f"[C3] Cardinality >5 violations:  {card_max_viol}")

# 4. Cardinality min
card_min_viol = (is_invested.sum(axis=1) < 2).sum()
print(f"[C4] Cardinality <2 violations:  {card_min_viol}")

# 5. Sector cap (40%)
print(f"[C5] Sector >40% violations:")
for sector, tks in sector_groups.items():
    valid = [t for t in tks if t in res_df.columns]
    if len(valid) > 1:
        sec_wt = res_df[valid].sum(axis=1)
        n_viol = (sec_wt > 0.401).sum()
        print(f"       {sector}: {n_viol}")

# 6. Defensive floor
def_tickers = sector_groups.get("Healthcare", []) + sector_groups.get("Consumer Staples", [])
def_tickers = [t for t in def_tickers if t in res_df.columns]
defensive_held = is_invested[def_tickers].sum(axis=1) >= 1
def_viol = (~defensive_held).sum()
print(f"[C6] No defensive stock:         {def_viol}")

# 7. Tech → Financials contingency
tech_held = is_invested[sector_groups.get("Technology", [])].any(axis=1)
fin_held  = is_invested[sector_groups.get("Financials",  [])].any(axis=1)
contingency_viol = (tech_held & ~fin_held).sum()
print(f"[C7] Tech w/o Financials:        {contingency_viol}")

# 8. XOM + CVX mutual exclusion
if "XOM" in res_df.columns and "CVX" in res_df.columns:
    mutex_viol = (is_invested["XOM"] & is_invested["CVX"]).sum()
    print(f"[C8] XOM+CVX both held:          {mutex_viol}")

print()
active_weights = res_df[TICKERS].values[res_df[TICKERS].values > epsilon]
if len(active_weights):
    print(f"   Active weight range:  {active_weights.min():.3f} – {active_weights.max():.3f}")
print(f"   Mean weight sum:      {weight_sums.mean():.4f}")
print()
print("   All zeros = all constraints satisfied across the frontier.")
print("=" * 55)


In [ ]:
# Full allocation table sorted by risk
display_cols = TICKERS + ['Return', 'N_Stocks']
res_display = res_df[display_cols].copy()
res_display.index = res_display.index.map(lambda x: f"{x:.4f}")
res_display.index.name = "Risk Limit"
res_display[TICKERS] = res_display[TICKERS].applymap(lambda v: f"{v:.3f}" if v > 1e-4 else "—")
res_display['Return'] = res_display['Return'].apply(lambda v: f"{float(v):.4f}")
print(res_display.to_string())


## What This Demonstrates

**Integer constraints create a fundamentally different frontier.**

In continuous MPT, the efficient frontier is a smooth curve — you can always
marginally adjust weights to improve the risk/return tradeoff. Here, the integer
constraints force discrete jumps. You can see this in the cardinality plot above:
the number of stocks held changes in steps, not gradually. Each step corresponds
to a point where the optimizer found it worthwhile to add (or drop) an entire position.

**The sector constraints are the most operationally realistic piece.**
Real institutional mandates often look exactly like this: must hold at least one defensive
sector, no single sector exceeds 40%, and correlated pairs (XOM/CVX) are mutually exclusive
because holding both gives you concentrated factor exposure without meaningful diversification benefit.

**The Tech → Financials contingency** mirrors how macro risk analysts think:
technology companies carry significant credit and liquidity dependencies. Pairing them
with a Financials position provides a partial natural hedge at the factor level.

**CV framing:**
This is a demonstrably harder problem class than the continuous MPT you may have seen
in other courses. Combining MINLP (Bonmin), live market data (yfinance), dynamic constraint
generation (sector_groups), and a full validation suite in one notebook shows end-to-end
quantitative workflow skill — data acquisition, model formulation, integer programming,
and result interpretation. That's the combination that matters in quant finance and
enterprise analytics roles.


---
## Step 9: Optimal Portfolio — Maximum Sharpe Ratio Point

The "top-left" of an efficient frontier is the portfolio with the best
**return per unit of risk** — formally, the maximum Sharpe ratio.

In a MINLP frontier the curve has discrete jumps (integer effects), so we
can't just take a derivative. Instead we compute the Sharpe ratio for every
feasible point in `res_df` and pick the argmax.

We use a **risk-free rate of 0% monthly** (conservative; adjust via
`RF_MONTHLY` below). The formula is:

```
Sharpe = (Portfolio Return − Risk-Free Rate) / sqrt(Portfolio Variance)
```

This cell reads `res_df` produced by the frontier sweep — nothing else changes.


In [ ]:
# ── Maximum Sharpe Ratio Selection ────────────────────────────────────────────
# Assumes res_df exists from Step 6.
# Adjust RF_MONTHLY if you want a non-zero risk-free baseline.

RF_MONTHLY = 0.0  # monthly risk-free rate (0% = conservative default)
                   # e.g. ~4.5% annual → 0.045/12 ≈ 0.00375

if len(res_df) == 0:
    print("No frontier data — run Step 6 first.")
else:
    # res_df index IS the risk limit (variance), Return column is expected return
    sharpe = (res_df['Return'] - RF_MONTHLY) / np.sqrt(res_df.index.astype(float))
    best_idx  = sharpe.idxmax()           # risk level of optimal point
    best_row  = res_df.loc[best_idx]
    best_sharpe = sharpe[best_idx]

    print("=" * 52)
    print("   MAXIMUM SHARPE RATIO PORTFOLIO")
    print("=" * 52)
    print(f"   Risk (variance):     {best_idx:.5f}")
    print(f"   Std Dev:             {np.sqrt(best_idx):.5f}")
    print(f"   Expected Return:     {best_row['Return']:.5f}  ({best_row['Return']*100:.3f}% / month)")
    print(f"   Sharpe Ratio:        {best_sharpe:.4f}")
    print(f"   Stocks held:         {int(best_row['N_Stocks'])}")
    print()
    print("   Allocation:")
    for t in TICKERS:
        w = best_row[t]
        if w > 1e-4:
            bar = "█" * int(w * 40)
            print(f"     {t:6s}  {w:.3f}  {bar}")
    print("=" * 52)


In [ ]:
# ── Re-plot Efficient Frontier with Sharpe Optimal Point Highlighted ──────────

if len(res_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))

    # Full frontier
    sc = ax.scatter(
        res_df.index, res_df['Return'],
        c=res_df['N_Stocks'], cmap='plasma', s=55, zorder=4, alpha=0.85
    )
    ax.plot(res_df.index, res_df['Return'], 'k-', alpha=0.2, linewidth=1)
    plt.colorbar(sc, ax=ax, label='Stocks Held')

    # Sharpe-optimal point
    ax.scatter(
        best_idx, best_row['Return'],
        s=220, color='red', zorder=6, marker='*', label=f'Max Sharpe ({best_sharpe:.3f})'
    )
    ax.annotate(
        f"  Max Sharpe\n  Return={best_row['Return']:.4f}\n  Risk={best_idx:.5f}",
        xy=(best_idx, best_row['Return']),
        xytext=(best_idx + (res_df.index.max() - res_df.index.min()) * 0.08,
                best_row['Return'] - (res_df['Return'].max() - res_df['Return'].min()) * 0.12),
        fontsize=8.5,
        arrowprops=dict(arrowstyle='->', color='red', lw=1.4),
        color='red'
    )

    # Sharpe ratio curve on secondary axis
    ax2 = ax.twinx()
    sharpe_vals = (res_df['Return'] - RF_MONTHLY) / np.sqrt(res_df.index.astype(float))
    ax2.plot(res_df.index, sharpe_vals, '--', color='steelblue', alpha=0.5, linewidth=1.3,
             label='Sharpe ratio')
    ax2.set_ylabel('Sharpe Ratio', color='steelblue', fontsize=9)
    ax2.tick_params(axis='y', labelcolor='steelblue')

    ax.set_title("Efficient Frontier — Max Sharpe Point (★) + Sharpe Curve", fontsize=12)
    ax.set_xlabel("Portfolio Risk (Variance)")
    ax.set_ylabel("Expected Monthly Return")
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(True, alpha=0.18)

    plt.tight_layout()
    plt.show()


---
## Step 10: Daily 3:55 PM Price Update — GitHub Actions Architecture

The scheduler does **not** run inside this notebook.  
It runs as a free GitHub Actions cron job that fires at 3:55 PM ET every weekday,
appends one row to `data/prices_3pm55.csv`, and commits it back to the repo automatically.

```
GitHub Actions (.github/workflows/price_capture.yml)
    └── runs capture_prices.py at 3:55 PM ET (Mon–Fri)
        └── appends row to data/prices_3pm55.csv
            └── commits + pushes to repo

This notebook (Colab or local)
    └── reads data/prices_3pm55.csv from the repo
        └── rebuilds returns_df, df_return, df_cov
            └── re-run Steps 6–9 → fresh frontier
```

**Why this is better than a Colab thread:**
- Colab sessions time out. GitHub Actions doesn't.
- The CSV grows in your repo — it becomes your permanent dataset.
- You can see every capture in the repo's Actions tab with full logs.
- Zero cost for public repos. Free tier is generous for private repos too.

**Setup steps (one time):**
1. Push this repo to GitHub (your school account works fine)
2. Go to `Settings → Actions → General` → confirm Actions are enabled
3. That's it. The workflow triggers automatically on schedule.

**To test manually:** Go to `Actions` tab → `Daily 3:55 PM Price Capture` → `Run workflow`.


### Reading the CSV in Colab

Two options depending on whether your repo is public or private.


In [ ]:
# ── Option A: Public repo — read CSV directly via raw URL ─────────────────────
# Replace with your actual GitHub username and repo name.

GITHUB_USER = "your-github-username"   # ← change this
GITHUB_REPO = "your-repo-name"         # ← change this
BRANCH      = "main"

RAW_CSV_URL = (
    f"https://raw.githubusercontent.com/"
    f"{GITHUB_USER}/{GITHUB_REPO}/{BRANCH}/data/prices_3pm55.csv"
)

print(f"Will read from:\n  {RAW_CSV_URL}")


In [ ]:
# ── Option B: Private repo — mount Google Drive and clone via token ───────────
# Only needed if your repo is private. Skip if public.
#
# Step 1: Create a GitHub Personal Access Token (PAT):
#   GitHub → Settings → Developer Settings → Personal Access Tokens → Fine-grained
#   Permissions needed: Contents (read)
#
# Step 2: Save the token in a Google Drive text file (never paste it raw in a notebook)
#   e.g. save as MyDrive/github_token.txt
#
# Step 3: Uncomment and run the cell below.

# from google.colab import drive
# drive.mount('/content/drive')
#
# with open('/content/drive/MyDrive/github_token.txt') as f:
#     GH_TOKEN = f.read().strip()
#
# GITHUB_USER = "your-github-username"
# GITHUB_REPO = "your-repo-name"
# BRANCH      = "main"
#
# RAW_CSV_URL = (
#     f"https://{GH_TOKEN}@raw.githubusercontent.com/"
#     f"{GITHUB_USER}/{GITHUB_REPO}/{BRANCH}/data/prices_3pm55.csv"
# )


In [ ]:
# ── Load the price log and rebuild returns ────────────────────────────────────
# Works for both Option A (public) and Option B (private) — RAW_CSV_URL is the same.

import pandas as pd
import numpy as np

def rebuild_returns_from_github(url=RAW_CSV_URL):
    """
    Reads prices_3pm55.csv from the GitHub repo and rebuilds
    returns_df, df_return, df_cov globals used by the MINLP solver.

    Call this at the start of any Colab session to pick up the latest data.
    """
    global returns_df, df_return, df_cov

    try:
        prices = pd.read_csv(url, index_col=0, parse_dates=True)
    except Exception as e:
        print(f"Could not read CSV from GitHub: {e}")
        print("Check that GITHUB_USER and GITHUB_REPO are set correctly above.")
        return None

    prices = prices.sort_index()[TICKERS]

    if len(prices) < 2:
        print(f"Only {len(prices)} rows in log — need at least 2 trading days.")
        print("The Actions workflow may not have run yet. Check the Actions tab.")
        return None

    returns_log = prices.pct_change().dropna()

    # Update globals — solver cells read these directly
    returns_df = returns_log
    df_return  = returns_log.mean()
    df_cov     = returns_log.cov()

    print(f"returns_df rebuilt from GitHub CSV:")
    print(f"  Trading days in log : {len(returns_log)}")
    print(f"  Date range          : {returns_log.index[0].date()} → {returns_log.index[-1].date()}")
    print(f"  Tickers             : {list(returns_log.columns)}")
    print()
    print("Re-run Steps 6–9 to solve the frontier with updated data.")
    return returns_log

# ── Call it ───────────────────────────────────────────────────────────────────
rebuild_returns_from_github()


### Monthly resampling (after ~22 trading days)

Once you have roughly a month of daily 3:55 prices in the log, you can
compound them into monthly returns — making them directly comparable to the
Womack dataset and standard MPT literature. Add this block after `rebuild_returns_from_github()`:

```python
# Compound daily returns → monthly
returns_monthly = (1 + returns_df).resample('ME').prod() - 1
returns_df = returns_monthly
df_return  = returns_monthly.mean()
df_cov     = returns_monthly.cov()
print(f"Resampled to {len(returns_monthly)} monthly periods.")
```

### How the Actions tab looks when it's working

Every successful run appears as a green checkmark under  
`Actions → Daily 3:55 PM Price Capture`.  
Click any run to see the exact prices captured and which row was appended.  
A red ✗ means the market was closed or yfinance had a hiccup — check the log.

### Note on the dual cron entries

The workflow has two cron triggers:
- `55 19 * * 1-5` → 3:55 PM ET during summer (EDT = UTC−4)
- `55 20 * * 1-5` → 3:55 PM ET during winter (EST = UTC−5)

GitHub Actions runs on UTC with no concept of daylight saving. The capture
script's holiday check and duplicate-row guard mean the second trigger on
any given day is always a safe no-op — it just exits cleanly.
